## Parameters

In [ ]:
SOURCE_DB  = "Sandbox_LH_Bronze"             # Bronze lakehouse SQL DB name
SOURCE_SCHEMA = "blog"                   # Bronze lakehouse SQL DB name
SOURCE_TABLE = "FabricUpdates"        # Bronze table with last ~14 days

DESTINATION_DB  = "Sandbox_LH_Silver"        # Silver lakehouse SQL DB name
DESTINATION_SCHEMA = "blog"
DESTINATION_TABLE = "FabricUpdates"   # Silver accumulated history table

CREATE_DESTINATION_TABLE = False        #Set to true on initial run, or when doing a complete reset/rebuild and blowing away the history

DEBUG_MODE = False                      # Set to False to execute the MERGE

## Notebook level Libraries, Settings, and Variables

In [ ]:
from pyspark.sql.functions import col
from pyspark.sql import functions as F
from pyspark.sql.utils import AnalysisException

# Setting Spark SQL casesensitive to "True" allows for case sensitive table names
spark.conf.set("spark.sql.caseSensitive", True)

if SOURCE_TABLE == "FabricUpdates":
    KEY_COLS = ["Url"]          # Unique key column Demographics
if SOURCE_TABLE == "blah":
    KEY_COLS = ["blahblah"]                # Unique key column History

# Database, table, and column variables
source_full = f"`{SOURCE_DB}`.`{SOURCE_SCHEMA}`.`{SOURCE_TABLE}`"
destination_full = f"`{DESTINATION_DB}`.`{DESTINATION_SCHEMA}`.`{DESTINATION_TABLE}`"
###key_col_sql = f"`{KEY_COL}`"  # for SQL fragments  # Running some new code later to handle the Python List


# Helpers for quoting/qualification
def quote_ident(name: str) -> str:
    return f"`{name.replace('`','``')}`"

def qual_namespace(db: str, schema: str) -> str:
    return f"{quote_ident(db)}.{quote_ident(schema)}"     # db.schema

def qual_table(db: str, schema: str, table: str) -> str:
    return f"{quote_ident(db)}.{quote_ident(schema)}.{quote_ident(table)}"  # db.schema.table


In [ ]:
print(source_full)
print(destination_full)
#print(key_col_sql)

## Load and Process Tables

In [ ]:
# Load tables
df_source      = spark.table(source_full)
df_destination = spark.table(destination_full)

## Create Destination Table on Initial Run

In [ ]:
if CREATE_DESTINATION_TABLE:
    def namespace_exists(db: str, schema: str) -> bool:
        """
        Reliable, schema-aware check: try SHOW TABLES IN db.schema.
        If the namespace is wrong, Spark will throw; we catch and return False.
        """
        try:
            _ = spark.sql(f"SHOW TABLES IN {qual_namespace(db, schema)}").limit(1).collect()
            return True
        except Exception:
            return False

    def table_exists(db: str, schema: str, table: str) -> bool:
        """
        Most reliable: try to resolve the table to a DataFrame.
        """
        try:
            _ = spark.table(qual_table(db, schema, table)).limit(0).collect()
            return True
        except AnalysisException:
            return False
        except Exception:
            # Any other resolution error => treat as not found
            return False

    src_ns = qual_namespace(SOURCE_DB, SOURCE_SCHEMA)
    dst_ns = qual_namespace(DESTINATION_DB, DESTINATION_SCHEMA)
    src_fq = qual_table(SOURCE_DB, SOURCE_SCHEMA, SOURCE_TABLE)
    dst_fq = qual_table(DESTINATION_DB, DESTINATION_SCHEMA, DESTINATION_TABLE)

    # ---- Fail fast on typo / missing namespace ----------------------------------
    if not namespace_exists(SOURCE_DB, SOURCE_SCHEMA):
        raise ValueError(f"Source namespace does not exist: {src_ns} "
                        "(is the HR_LH_Bronze lakehouse attached and schema correct?)")
    if not table_exists(SOURCE_DB, SOURCE_SCHEMA, SOURCE_TABLE):
        raise ValueError(f"Source table does not exist: {src_fq}")

    if not namespace_exists(DESTINATION_DB, DESTINATION_SCHEMA):
        raise ValueError(f"Destination namespace does not exist: {dst_ns} "
                        "(attach HR_LH_Silver and verify schema name)")

    # ---- Create destination table if missing (clone schema) ----------------------
    if table_exists(DESTINATION_DB, DESTINATION_SCHEMA, DESTINATION_TABLE):
        print(f"Destination table already exists: {dst_fq}")
        display(spark.table(dst_fq).limit(0))
    else:
        df_source = spark.table(src_fq)
        empty_df  = spark.createDataFrame(spark.sparkContext.emptyRDD(), df_source.schema)

        writer = (empty_df.write
                .format("delta")
                .mode("overwrite")
                .option("delta.columnMapping.mode", "name"))
        # if ENABLE_COLUMN_MAPPING:
        #     writer = (writer
        #               .option("delta.columnMapping.mode", "name")
        #               .option("delta.minReaderVersion", "2")
        #               .option("delta.minWriterVersion", "5"))

        writer.saveAsTable(dst_fq)

        print(f"Created empty Delta table: {dst_fq}")
        display(spark.table(dst_fq).limit(0))
        print("Source dtypes      ->", df_source.dtypes)
        print("Destination dtypes ->", spark.table(dst_fq).dtypes)


In [ ]:
# Build quoted key list for SQL
key_cols_sql = [quote_ident(c) for c in KEY_COLS]

# Basic checks: keys present on both sides
missing_src = [c for c in KEY_COLS if c not in df_source.columns]
missing_dst = [c for c in KEY_COLS if c not in df_destination.columns]
if missing_src:
    raise ValueError(f"Source is missing key column(s): {missing_src}")
if missing_dst:
    raise ValueError(f"Destination is missing key column(s): {missing_dst}")

# Enforce uniqueness across the composite key in source
dups = (df_source.groupBy(*KEY_COLS).count().filter("count > 1").limit(1).count())
if dups > 0:
    raise ValueError(f"Source has duplicate key(s) for columns {KEY_COLS}; dedupe before merge.")

# Optional: disallow NULL in any key column
null_cond = None
for c in KEY_COLS:
    cond = F.col(c).isNull()
    null_cond = cond if null_cond is None else (null_cond | cond)
null_keys = df_source.filter(null_cond).limit(1).count()
if null_keys > 0:
    raise ValueError(f"Source has NULL in one or more key columns {KEY_COLS}; fix upstream before merge.")

In [ ]:
# Align to destination schema (preserve destination column order)
common_cols = [c for c in df_destination.columns if c in df_source.columns]
update_cols = [c for c in common_cols if c not in KEY_COLS]

# Preview impact
destination_keys = df_destination.select(*KEY_COLS).distinct()
matched_df  = df_source.join(destination_keys, on=KEY_COLS, how="inner")      # will UPDATE
new_df      = df_source.join(destination_keys, on=KEY_COLS, how="left_anti") # will INSERT

matched_cnt = matched_df.count()
new_cnt     = new_df.count()

print("=== Upsert Plan Preview ===")
print(f"Source      : {source_full}")
print(f"Destination : {destination_full}")
print(f"Keys        : {KEY_COLS}")
print(f"Matched in Destination : {matched_cnt}  (will be UPDATED)")
print(f"New to Destination     : {new_cnt}      (will be INSERTED)")
print("\nSample matched keys:")
display(matched_df.select(*KEY_COLS).limit(10))
print("Sample new keys:")
display(new_df.select(*KEY_COLS).limit(10))

In [ ]:
# Compose MERGE SQL
on_clause  = " AND ".join([f"S.{k} <=> B.{k}" for k in key_cols_sql])  # null-safe equality
update_set = ", ".join([f"S.{quote_ident(c)} = B.{quote_ident(c)}" for c in update_cols]) if update_cols else None
insert_cols = ", ".join([quote_ident(c) for c in common_cols])
insert_vals = ", ".join([f"B.{quote_ident(c)}" for c in common_cols])

merge_sql = f"""
MERGE INTO {destination_full} AS S
USING {source_full} AS B
ON {on_clause}
"""
if update_set:
    merge_sql += f"WHEN MATCHED THEN UPDATE SET {update_set}\n"
merge_sql += f"WHEN NOT MATCHED THEN INSERT ({insert_cols}) VALUES ({insert_vals})"

print("\n=== MERGE statement ===")
print(merge_sql)



In [ ]:
if DEBUG_MODE:
    print("\nDEBUG_MODE=True -> No MERGE executed.")
else:
    print("\nExecuting MERGE ...")
    spark.sql(merge_sql)
    print("MERGE completed.")

    # Optional quick post-check
    post = (
        spark.table(source_full)
        .join(spark.table(destination_full).select(*KEY_COLS).distinct(),
              on=KEY_COLS, how="inner")
        .count()
    )
    print(f"Post-check: Source keys now present in Destination = {post}")